# Currency Conversion Tool

In [ ]:
from langchain_huggingface import ChatHuggingFace
from langchain_cohere import ChatCohere
from langchain_core.messages import HumanMessage
import requests
from langchain_core.tools import tool

In [ ]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

api_key = "002208535bf3b9c8f9835a3a"
base_currency = "USD"
target_currency = "NPR"

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """This function fetches the currency factor between a given base currency and a target currency"""
    url = f'https://v6.exchangerate-api.com/v6/002208535bf3b9c8f9835a3a/pair/{base_currency}/{target_currency}'

    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()
        print(f"1 {base_currency} = {data['conversion_rate']} {target_currency}")
    else:
        print(f"Error {response.status_code}: Could not fetch rates.")
    return response.json()    

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """given the currency conversion rate this function calculates the target currency value from a given base currency value"""

    return base_currency_value * conversion_rate



In [ ]:
convert.args

In [ ]:
get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency':'NPR'})

In [ ]:
convert.invoke({'base_currency_value': 10, 'conversion_rate':153.2231 })

In [ ]:
# tool binding

llm = ChatCohere()

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
messages = [HumanMessage('What is the conversion factor between USD and NPR, and based on that can you convert 10 usd to npr')]

In [ ]:
messages

In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
messages.append(ai_message)

In [ ]:
ai_message.tool_calls

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
import json

for tool_call in ai_message.tool_calls:
    # execute the 1st tool and get the value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        print(tool_message1)
        # fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this tool message to messages list
        messages.append(tool_message1)
    #  execute the 2nd tool using the conversion rate from tool 1    
    if tool_call['name'] == 'convert':
        # fetch the current argument
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_messages2 = convert.invoke(tool_call)
        messages.append(tool_messages2)


In [ ]:
llm_with_tools.invoke(messages)